# Notebook 06 v2 — Model C: Chunk-Based BERT + Label Attention (Fixed)

**v2 fixes over v1:**
1. **chunk_counts now properly masks padding chunks** (was ignored before)
2. **Focal loss** replaces BCEWithLogitsLoss + pos_weight=50 (fixes calibration)
3. **Validation F1 now computed with threshold tuning** (was hardcoded 0.5)
4. **Per-label linear classifier** replaces dot-product logit (more expressive)
5. **ICD code description embeddings** initialize label queries semantically
6. **7 fine-tune epochs** with early stopping (was 3, too few)
7. **Temperature scaling** for post-hoc calibration

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

from src.config import (
    DATA_DIR, MODEL_C_DIR, TRANSFORMER_MODEL, MAX_SEQ_LEN, SEED,
    MODEL_C_MAX_CHUNKS, MODEL_C_CHUNK_STRIDE,
    MODEL_C_FROZEN_LR, MODEL_C_FINETUNE_LR,
    MODEL_C_FROZEN_EPOCHS, MODEL_C_FINETUNE_EPOCHS,
    MODEL_C_BATCH_SIZE, MODEL_C_GRAD_ACCUM, MODEL_C_DROPOUT,
    MODEL_C_FOCAL_GAMMA, MODEL_C_FOCAL_ALPHA, MODEL_C_EARLY_STOP,
    USE_AMP,
)
from src.train import set_seed

# Use a separate directory so v1 results are preserved
SAVE_DIR = MODEL_C_DIR / 'v2'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    print(f'CUDA: {torch.version.cuda}')
    print(f'fp16: ENABLED')
else:
    print('WARNING: No GPU!')

print(f'\nv2 Config:')
print(f'  Focal loss:      gamma={MODEL_C_FOCAL_GAMMA}, alpha={MODEL_C_FOCAL_ALPHA}')
print(f'  Frozen epochs:   {MODEL_C_FROZEN_EPOCHS}')
print(f'  Finetune epochs: {MODEL_C_FINETUNE_EPOCHS} (early stop patience={MODEL_C_EARLY_STOP})')
print(f'  Batch:           {MODEL_C_BATCH_SIZE} x {MODEL_C_GRAD_ACCUM} = {MODEL_C_BATCH_SIZE * MODEL_C_GRAD_ACCUM}')

## 1. Load Data

In [ ]:
from src.data import load_splits, load_label_binarizer, build_label_matrix, get_icd_descriptions

train_df, val_df, test_df = load_splits()
mlb = load_label_binarizer()
vocab = list(mlb.classes_)
NUM_LABELS = len(vocab)

Y_train = build_label_matrix(train_df, mlb)
Y_val   = build_label_matrix(val_df, mlb)
Y_test  = build_label_matrix(test_df, mlb)

print(f'Labels: {NUM_LABELS}')
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

## 2. Create Datasets & DataLoaders

In [ ]:
from src.data import ChunkedICDDataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)

train_ds = ChunkedICDDataset(train_df['clean_text'], Y_train, tokenizer=tokenizer)
val_ds   = ChunkedICDDataset(val_df['clean_text'],   Y_val,   tokenizer=tokenizer)
test_ds  = ChunkedICDDataset(test_df['clean_text'],  Y_test,  tokenizer=tokenizer)

train_loader = DataLoader(train_ds, batch_size=MODEL_C_BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=MODEL_C_BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=MODEL_C_BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

# Verify chunking
sample = train_ds[0]
print(f'Sample: input_ids={sample["input_ids"].shape}, chunks={sample["chunk_count"]}')

## 3. Build Model C v2 with Code Description Embeddings

In [ ]:
from src.models import LabelAttentionClassifier

model = LabelAttentionClassifier(
    model_name=TRANSFORMER_MODEL,
    num_labels=NUM_LABELS,
    max_chunks=MODEL_C_MAX_CHUNKS,
    freeze_bert=True,
    dropout=MODEL_C_DROPOUT,
).to(device)

# Initialize label queries from ICD code descriptions (TIER 2 improvement)
descriptions = get_icd_descriptions(vocab)
print('\nSample descriptions:')
for i in range(min(5, len(vocab))):
    print(f'  {vocab[i]:10s} -> {descriptions[i]}')

model.init_label_queries_from_descriptions(descriptions, tokenizer=tokenizer, device=device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters()) - trainable
print(f'\nTrainable: {trainable:,}  Frozen: {frozen:,}')
print(f'BERT: FROZEN (Phase 1)')

## 4. Phase 1 — Frozen BERT with Focal Loss
Using focal loss (gamma=2.0) instead of BCE+pos_weight=50. Focal loss handles class imbalance without distorting probability calibration.

In [ ]:
from src.train import train_model

print('='*60)
print('PHASE 1: Frozen BERT + Focal Loss + Code Description Init')
print('='*60)

history_frozen = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=SAVE_DIR,
    lr=MODEL_C_FROZEN_LR,
    epochs=MODEL_C_FROZEN_EPOCHS,
    grad_accum=MODEL_C_GRAD_ACCUM,
    warmup_ratio=0.05,
    use_focal_loss=True,
    focal_gamma=MODEL_C_FOCAL_GAMMA,
    focal_alpha=MODEL_C_FOCAL_ALPHA,
    checkpoint_every=5000,
    is_chunked=True,
    device=device,
)

## 5. Phase 2 — Fine-tune Last 2 BERT Layers with Early Stopping

In [ ]:
# Reload best Phase 1 checkpoint
model.load_state_dict(torch.load(SAVE_DIR / 'best_model.pt', map_location=device))
model.unfreeze_bert_layers(num_layers=2)

print('\n' + '='*60)
print('PHASE 2: Fine-tune + Focal Loss + Early Stopping')
print('='*60)

history_finetune = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir=SAVE_DIR,
    lr=MODEL_C_FINETUNE_LR,
    epochs=MODEL_C_FINETUNE_EPOCHS,
    grad_accum=MODEL_C_GRAD_ACCUM,
    warmup_ratio=0.1,
    use_focal_loss=True,
    focal_gamma=MODEL_C_FOCAL_GAMMA,
    focal_alpha=MODEL_C_FOCAL_ALPHA,
    checkpoint_every=5000,
    is_chunked=True,
    early_stopping_patience=MODEL_C_EARLY_STOP,
    device=device,
)

## 6. Training History

In [ ]:
import matplotlib.pyplot as plt

full_history = history_frozen + [
    {**h, 'epoch': h['epoch'] + MODEL_C_FROZEN_EPOCHS} for h in history_finetune
]
hist_df = pd.DataFrame(full_history)
hist_df.to_csv(SAVE_DIR / 'full_training_history.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'b-o', markersize=4)
axes[0].axvline(x=MODEL_C_FROZEN_EPOCHS + 0.5, color='red', linestyle='--', alpha=0.5, label='Unfreeze')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Train Loss')
axes[0].legend()

# Val F1 (tuned vs default)
axes[1].plot(hist_df['epoch'], hist_df['val_f1_tuned'], 'g-o', markersize=4, label='Tuned threshold')
axes[1].plot(hist_df['epoch'], hist_df['val_f1_at_0.5'], 'gray', linestyle='--', alpha=0.5, label='F1 @ t=0.5')
axes[1].axvline(x=MODEL_C_FROZEN_EPOCHS + 0.5, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Micro-F1'); axes[1].set_title('Validation F1')
axes[1].legend()

# Threshold evolution
axes[2].plot(hist_df['epoch'], hist_df['best_threshold'], 'r-o', markersize=4)
axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='t=0.5')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Threshold'); axes[2].set_title('Optimal Threshold')
axes[2].legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_curves_v2.png', dpi=150)
plt.show()
print(hist_df.to_string(index=False))

## 7. Temperature Scaling (Calibration)

In [ ]:
from src.models import TemperatureScaler
from src.train import collect_logits, evaluate_predictions
from src.evaluate import tune_global_threshold, expected_calibration_error

# Reload best model
model.load_state_dict(torch.load(SAVE_DIR / 'best_model.pt', map_location=device))
model.eval()

# Collect raw logits on validation set
print('Collecting validation logits...')
val_logits, val_labels = collect_logits(
    model, val_loader, device=device, use_amp=USE_AMP, is_chunked=True,
)

# Fit temperature scaler
scaler = TemperatureScaler()
T = scaler.fit(val_logits, val_labels)

# Compare calibration before and after
import torch as th
probs_uncal = th.sigmoid(th.tensor(val_logits)).numpy()
probs_cal   = scaler.calibrate(val_logits)

ece_before = expected_calibration_error(probs_uncal, val_labels)
ece_after  = expected_calibration_error(probs_cal, val_labels)

t_before, f1_before = tune_global_threshold(probs_uncal, val_labels)
t_after, f1_after   = tune_global_threshold(probs_cal, val_labels)

print(f'\nCalibration Results:')
print(f'  Before: ECE={ece_before:.4f}  best_t={t_before:.3f}  F1={f1_before:.4f}')
print(f'  After:  ECE={ece_after:.4f}  best_t={t_after:.3f}  F1={f1_after:.4f}')
print(f'  Temperature: {T:.4f}')

# Save temperature
import json
with open(SAVE_DIR / 'temperature.json', 'w') as f:
    json.dump({'temperature': T, 'ece_before': ece_before, 'ece_after': ece_after}, f, indent=2)

## 8. Test Evaluation (with calibration + per-label thresholds)

In [ ]:
from src.evaluate import full_metrics, tune_per_label_threshold
import json

# Test predictions
print('Computing test predictions...')
test_logits, Y_test_np = collect_logits(
    model, test_loader, device=device, use_amp=USE_AMP, is_chunked=True,
)

# Uncalibrated probabilities
P_test_uncal = th.sigmoid(th.tensor(test_logits)).numpy()
# Calibrated probabilities
P_test_cal = scaler.calibrate(test_logits)

# Global threshold (on calibrated probs)
best_t_global, _ = tune_global_threshold(probs_cal, val_labels)
results_global = full_metrics(P_test_cal, Y_test_np, best_t_global, 'Model C v2 (global threshold)')

# Per-label thresholds (on calibrated probs)
per_label_t = tune_per_label_threshold(probs_cal, val_labels)
preds_per_label = np.zeros_like(P_test_cal, dtype=int)
for j in range(P_test_cal.shape[1]):
    preds_per_label[:, j] = (P_test_cal[:, j] >= per_label_t[j]).astype(int)

from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
mask = Y_test_np.sum(0) > 0
results_per_label = {
    'Model':       'Model C v2 (per-label threshold)',
    'Threshold':   'per-label',
    'Micro-F1':    round(f1_score(Y_test_np, preds_per_label, average='micro', zero_division=0), 4),
    'Macro-F1':    round(f1_score(Y_test_np, preds_per_label, average='macro', zero_division=0), 4),
    'Micro-Prec':  round(precision_score(Y_test_np, preds_per_label, average='micro', zero_division=0), 4),
    'Micro-Rec':   round(recall_score(Y_test_np, preds_per_label, average='micro', zero_division=0), 4),
    'Macro-AUPRC': round(average_precision_score(Y_test_np[:, mask], P_test_cal[:, mask], average='macro'), 4),
    'Micro-AUROC': round(roc_auc_score(Y_test_np[:, mask], P_test_cal[:, mask], average='micro'), 4),
}

print('\n=== Test Results (Global Threshold) ===')
for k, v in results_global.items():
    print(f'  {k:15s}: {v}')

print('\n=== Test Results (Per-Label Thresholds) ===')
for k, v in results_per_label.items():
    print(f'  {k:15s}: {v}')

# Save all results
with open(SAVE_DIR / 'test_results.json', 'w') as f:
    json.dump({'global_threshold': results_global, 'per_label_threshold': results_per_label}, f, indent=2)
np.save(SAVE_DIR / 'P_test_calibrated.npy', P_test_cal)
np.save(SAVE_DIR / 'P_test_uncalibrated.npy', P_test_uncal)
np.save(SAVE_DIR / 'per_label_thresholds.npy', per_label_t)
print('\nResults saved to', SAVE_DIR)

## 9. Head/Torso/Tail Analysis

In [ ]:
from sklearn.metrics import f1_score

per_label_f1 = f1_score(Y_test_np, preds_per_label, average=None, zero_division=0)
per_label_freq = Y_train.sum(0)

label_df = pd.DataFrame({
    'icd_code': vocab, 'train_freq': per_label_freq, 'test_f1': per_label_f1,
})

print(f'{"Bucket":25s}  {"n":>6}  {"Avg F1":>7}')
print('-' * 45)
for lo, hi, name in [(500, 1e9, 'head (>=500)'), (100, 499, 'torso (100-499)'), (0, 99, 'tail (<100)')]:
    s = label_df[(label_df['train_freq'] >= lo) & (label_df['train_freq'] <= hi)]
    print(f'{name:25s}  {len(s):6d}  {s["test_f1"].mean():7.4f}')

plt.figure(figsize=(7, 4))
plt.scatter(label_df['train_freq'].clip(upper=2000), label_df['test_f1'], alpha=0.3, s=8)
plt.xscale('log'); plt.xlabel('Train freq (log)'); plt.ylabel('Test F1')
plt.title('Per-label F1 vs training frequency (Model C v2)')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'head_tail_f1_v2.png', dpi=120)
plt.show()

## 10. Compare v1 vs v2

In [ ]:
# Load v1 results for comparison
from src.config import MODEL_A_DIR, MODEL_B_DIR

with open(MODEL_A_DIR / 'results.json') as f:
    res_a = json.load(f)
with open(MODEL_B_DIR / 'test_results.json') as f:
    res_b = json.load(f)

# v1 results
v1_results = None
try:
    with open(MODEL_C_DIR / 'test_results.json') as f:
        v1_results = json.load(f)
except FileNotFoundError:
    print('v1 results not found')

comparison = pd.DataFrame([
    {'Model': 'A: TF-IDF + SGD',
     'Micro-F1': res_a['test']['micro_f1'],
     'Macro-F1': res_a['test']['macro_f1'],
     'AUROC': res_a['test']['micro_auroc']},
    {'Model': 'B: ClinicalBERT',
     'Micro-F1': res_b['micro_f1'],
     'Macro-F1': res_b['macro_f1'],
     'AUROC': res_b['micro_auroc']},
])

if v1_results:
    comparison = pd.concat([comparison, pd.DataFrame([{
        'Model': 'C v1: Chunk+Attn (buggy)',
        'Micro-F1': v1_results.get('Micro-F1', 0),
        'Macro-F1': v1_results.get('Macro-F1', 0),
        'AUROC': v1_results.get('Micro-AUROC', 0),
    }])], ignore_index=True)

comparison = pd.concat([comparison, pd.DataFrame([{
    'Model': 'C v2: Fixed (global t)',
    'Micro-F1': results_global['Micro-F1'],
    'Macro-F1': results_global['Macro-F1'],
    'AUROC': results_global['Micro-AUROC'],
}, {
    'Model': 'C v2: Fixed (per-label t)',
    'Micro-F1': results_per_label['Micro-F1'],
    'Macro-F1': results_per_label['Macro-F1'],
    'AUROC': results_per_label['Micro-AUROC'],
}])], ignore_index=True)

print('\n=== All Models Comparison ===')
print(comparison.to_string(index=False))
comparison.to_csv(SAVE_DIR / 'comparison_all.csv', index=False)